## Feature Engineering

Import the required libraries

In [88]:
import pandas as pd
pd.set_option('display.max_columns', 100)


Import the cleaned dataset

In [89]:
df = pd.read_csv('flights_eda_df.csv')
df.sample(3)

,origin,City,destination,City_destination,Name_airline,aircraft,query_date,departure_date,departure_clock_time,day_of_week_departure,month_departure,arrival_date,arrival_clock_time,days_until_departure,trip_duration_minutes,number_of_stops,base_price,departure_hour,arrival_hour,departure_time_period,arrival_time_period,duration_group
2829,YOW,Ottawa,YYZ,Toronto,WestJet,7M8,2026-03-08,2026-06-19,06:00:00,Friday,6,2026-06-19,17:22:00,103,497,1,284.17,6,17,Early Morning,Evening,"[480, 540)"
36961,YYC,Calgary,YYZ,Toronto,Porter Airlines Canada Limited,295,2026-03-08,2026-03-22,12:40:00,Sunday,3,2026-03-22,18:33:00,14,233,0,113.04,12,18,Afternoon,Evening,"[180, 240)"
2916,YOW,Ottawa,YYZ,Toronto,Air Canada,223,2026-03-08,2026-06-23,10:00:00,Tuesday,6,2026-06-23,11:09:00,107,69,0,226.08,10,11,Morning,Morning,"[60, 120)"


In [90]:
df.shape

(43157, 22)

### I. Features to Create
- Weekend Flag: is_weekend = day_of_week_departure.isin(['Saturday', 'Sunday']).astype(int) (weekends often pricier).
- Season: season = month_departure.apply(lambda m: 'Winter' if m in [12,1,2] else 'Spring' if m in [3,4,5] else 'Summer' if m in [6,7,8] else 'Fall') (seasonal demand).

In [91]:
# creata a weekend feature
df['is_weekend_departure'] = df['day_of_week_departure'].isin(['Saturday', 'Sunday']).astype(int)

# create a route
df['route'] = df['origin'] + '-' + df['destination']

# seasonal feature
df['season'] = df['month_departure'].apply(lambda m: 'Winter' if m in [12,1,2] else 'Spring' if m in [3,4,5] else 'Summer' if m in [6,7,8] else 'Fall')

### II. Features Selection
#### Drop
- Redundant/High Correlation: City, City_destination (use origin/destination codes instead). month_departure (seasonal info better binned).
- Irrelevant/Noise: departure_clock_time, arrival_clock_time (already binned into periods). base_price if predicting total_price (avoid leakage).
- Low Impact: Name_airline outliers like First Air (already dropped). Any constant columns (e.g., if all flights are domestic).
- Post-EDA Drops: Temporary columns like route.

#### Keep
- Numerical: distance_km, days_until_departure, trip_duration_minutes, number_of_stops, bookable_seats (all correlate with price; bookable_seats indicates demand)
- Categorical: origin, destination, Name_airline, aircraft, day_of_week_departure, departure_time_period, arrival_time_period (capture route/airline/time effects).
- Target variable: base_price

In [92]:
# Using .loc to select the relevant features for modeling
df2 = df.loc[:,['base_price','origin','destination', 'departure_hour', 'arrival_hour',
       'Name_airline', 'day_of_week_departure', 'days_until_departure',
       'trip_duration_minutes', 'number_of_stops','departure_time_period',
       'arrival_time_period', 'is_weekend_departure', 'route',
       'season']]

df2.head(2)

,base_price,origin,destination,departure_hour,arrival_hour,Name_airline,day_of_week_departure,days_until_departure,trip_duration_minutes,number_of_stops,departure_time_period,arrival_time_period,is_weekend_departure,route,season
0,339.12,YOW,YYZ,5,7,WestJet,Monday,1,81,0,Early Morning,Early Morning,0,YOW-YYZ,Spring
1,339.12,YOW,YYZ,6,8,Porter Airlines Canada Limited,Monday,1,75,0,Early Morning,Morning,0,YOW-YYZ,Spring


### III. Dummy Variables

Use pandas .get_dummies to encode the categorical columns to numerical.

In [93]:
df3 = pd.get_dummies(df2, columns = ['origin', 'destination', 'Name_airline', 'day_of_week_departure',
       'departure_time_period', 'arrival_time_period', 'season','route'
       ], drop_first=True).astype(int)

df3.head(2)

,base_price,departure_hour,arrival_hour,days_until_departure,trip_duration_minutes,number_of_stops,is_weekend_departure,origin_YVR,origin_YYC,origin_YYZ,destination_YVR,destination_YYC,destination_YYZ,Name_airline_Air North Yukon's Airline,Name_airline_Air Transat,Name_airline_Central Mountain Air LTD,Name_airline_Pacific Coastal Airlines Limited,Name_airline_Porter Airlines Canada Limited,Name_airline_WestJet,day_of_week_departure_Monday,day_of_week_departure_Saturday,day_of_week_departure_Sunday,day_of_week_departure_Thursday,day_of_week_departure_Tuesday,day_of_week_departure_Wednesday,departure_time_period_Early Morning,departure_time_period_Evening,departure_time_period_Late Evening,departure_time_period_Morning,departure_time_period_Night,arrival_time_period_Early Morning,arrival_time_period_Evening,arrival_time_period_Late Evening,arrival_time_period_Morning,arrival_time_period_Night,season_Spring,season_Summer,season_Winter,route_YOW-YYC,route_YOW-YYZ,route_YVR-YOW,route_YVR-YYC,route_YVR-YYZ,route_YYC-YOW,route_YYC-YVR,route_YYC-YYZ,route_YYZ-YOW,route_YYZ-YVR,route_YYZ-YYC
0,339,5,7,1,81,0,0,0,0,0,0,0,1,0,0,0,0,0,1,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0
1,339,6,8,1,75,0,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0


In [94]:
# export csv
df3.to_csv('final_df.csv', index=False)
print('File saved')

File saved
